In [3]:
import numpy as np
import pandas as pd

def create_block(block_name, n_trials, pc_ratio):
    """
    Generates a randomized block of Go/No-Go trials based on the target conflict ratio.
    """ 
    # 1.trial counts depends pc, changing pc to tweak probabilities
    n_pc = int(n_trials * pc_ratio)
    n_pi = n_trials - n_pc
    # Split PC -> GW&NAL
    n_gw = n_pc // 2
    n_nal = n_pc - n_gw
    # Split PI -> NW&GAL
    n_nw = n_pi // 2
    n_gal = n_pi - n_nw
    
    trials = [] # empty dict, to which we attach the keys(block, Pav, trial, cue, action)
    # and values
    # 2. PC trials - cues and the correct actions
    # GW: cue(+1), action= go(1)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'GW', 
                    'Cue_Valence': 1, 'Optimal_Action': 1}] * n_gw)
    # NAL: cue(-1), action=No-Go (0)
    trials.extend([{'Block': block_name, 'Conflict': 'PC', 'Trial_Type': 'NAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 0}] * n_nal)
    
    # 3. PI trials
    # NW: cue(+1), action= nogo(0)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'NW', 
                     'Cue_Valence': 1, 'Optimal_Action': 0}] * n_nw)
    # GAL:cue (-1), action= go(1)
    trials.extend([{'Block': block_name, 'Conflict': 'PI', 'Trial_Type': 'GAL', 
                    'Cue_Valence': -1, 'Optimal_Action': 1}] * n_gal)
    
    # 4. Shuffle the trials so they appear in a random order within the block
    df = pd.DataFrame(trials)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Generate the Full Experiment
np.random.seed(42) # For reproducibility

# Set how many trials you want per block (e.g., 40 trials * 4 blocks = 160 total trials)
TRIALS_PER_BLOCK = 5000

# Generate the 4 specific blocks
b1 = create_block('B1_MC', TRIALS_PER_BLOCK, pc_ratio=0.50)
b2 = create_block('B2_HC1', TRIALS_PER_BLOCK, pc_ratio=0.30)
b3 = create_block('B3_HC2', TRIALS_PER_BLOCK, pc_ratio=0.30)
b4 = create_block('B4_LC', TRIALS_PER_BLOCK, pc_ratio=0.70)

# Combine them sequentially
experiment_df = pd.concat([b1, b2, b3, b4], ignore_index=True)

# Add Trial Numbers and Placeholders for the Agent
experiment_df.insert(0, 'Trial', experiment_df.index + 1)
experiment_df['Agent_Choice'] = np.nan
experiment_df['Agent_Reward'] = np.nan


# Verification
print("--- Experiment Structure Verification ---")
# Count the PC vs PI trials in each block to prove the math is correct
summary = experiment_df.groupby(['Block', 'Conflict']).size().unstack(fill_value=0)
summary['% PC'] = (summary['PC'] / (summary['PC'] + summary['PI'])) * 100
print(summary)

print("\n--- First 10 Trials of Block 1 ---")
print(experiment_df.head(5))

# Save the pure environment
# experiment_df.to_csv('pure_task_environment.csv', index=False)

--- Experiment Structure Verification ---
Conflict    PC    PI  % PC
Block                     
B1_MC     2500  2500  50.0
B2_HC1    1500  3500  30.0
B3_HC2    1500  3500  30.0
B4_LC     3500  1500  70.0

--- First 10 Trials of Block 1 ---
   Trial  Block Conflict Trial_Type  Cue_Valence  Optimal_Action  \
0      1  B1_MC       PC        NAL           -1               0   
1      2  B1_MC       PI         NW            1               0   
2      3  B1_MC       PI         NW            1               0   
3      4  B1_MC       PC         GW            1               1   
4      5  B1_MC       PC         GW            1               1   

   Agent_Choice  Agent_Reward  
0           NaN           NaN  
1           NaN           NaN  
2           NaN           NaN  
3           NaN           NaN  
4           NaN           NaN  


In [4]:
from scipy.special import expit as inv_logit

params = {
    'xi': 0.1, #noise rate, randomness
    'ep': 0.15, # learning rate
    'b':  0.5, # go bias
    'pi': 1.2, # pavlovian bias
    'rho': 2.0 # reward sensitivity
}

def decide(params, qv_g, qv_ng, sv):
    # sv = stimulus value
    # ep = learning rate
    # rho = rew sensitivity
    # outcome = the rew received this trial (+1,0,1)

    # sv is the stimulus value of the cue itself.
    # It pulls towards Go via pi*sv. 
    #=> a combined effect of pavlovian bias and stimulus value
    # when the cue is avoid, it pulls towards Go via pi*sv.
    # qv = Q value of doing go on this cue
    # qv_ng = Q value of doing nogo on this cue.
    
    b=params['b']
    pi=params['pi']
    wv_g  = qv_g + b + pi * sv # go action weight
    wv_ng = qv_ng  # nogo action weight
    pGo = inv_logit(wv_g - wv_ng)
    pGo = pGo*(1-params['xi'])
    pGo = pGo+(params['xi']/2)
    return pGo
    
float(decide(params, qv_g=0, qv_ng=0.8, sv=0.8))
# setting bias(b) = 3 and sv(cue predicts rew) = 0.8
# we got a strong go bias, ie cue for rew+pav bias is high, means 
# more tendency to go 
# high value of qv_ng implies nogo is correct. 
# the agent will keep making error on conflict trials.

def update (params, sv, qv_g, qv_ng, outcome, pressed):
    sv = sv + params['ep']*(params['rho']*outcome - sv)
    #Q-value update lines
    if (pressed):
        qv_g = qv_g + params['ep']*(params['rho']*outcome-qv_g)
    else:
        qv_ng = qv_ng + params['ep']*(params['rho']*outcome-qv_ng)
    return qv_g, qv_ng, sv



In [5]:
def run_agent(experiment_df):
    # dictionary per cue
    cues=['GW','NAL','NW','GAL']
    qv_g  = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    qv_ng = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    sv    = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    for idx, row in experiment_df.iterrows():
        cue = row['Trial_Type']
        pGo=decide(params, qv_g[cue], qv_ng[cue], sv[cue])
        action=np.random.binomial(1,pGo)
        experiment_df.at[idx, 'Agent_Choice'] = action
        correct = (action == row['Optimal_Action'])
        if row['Cue_Valence'] == 1:
            reward = 1 if correct else 0
        else:
            reward = 0 if correct else -1
        experiment_df.at[idx, 'Agent_Reward'] = reward
        ## replaced outcome w rew and pressed w action
        qv_g[cue], qv_ng[cue], sv[cue] = update(params, sv[cue], qv_g[cue], qv_ng[cue], reward, action)
        
        

In [6]:
run_agent(experiment_df)
print(experiment_df[['Trial', 'Block', 'Trial_Type', 'Agent_Choice', 'Agent_Reward']].head(5))

   Trial  Block Trial_Type  Agent_Choice  Agent_Reward
0      1  B1_MC        NAL           1.0          -1.0
1      2  B1_MC         NW           0.0           1.0
2      3  B1_MC         NW           0.0           1.0
3      4  B1_MC         GW           0.0           0.0
4      5  B1_MC         GW           1.0           1.0


In [7]:
experiment_df['Correct'] = (experiment_df['Agent_Choice'] == experiment_df['Optimal_Action']).astype(int)
print(experiment_df.groupby('Trial_Type')['Correct'].mean().round(3))

Trial_Type
GAL    0.859
GW     0.941
NAL    0.827
NW     0.534
Name: Correct, dtype: float64


In [8]:
experiment_df.to_csv('simulated_agent_5k.csv', index=False)

In [9]:
# Parameter retrieval
from scipy.optimize import minimize

df = pd.read_csv('simulated_agent_5k.csv')
def compute_nll(params_list, df):
    xi, ep, b, pi, rho = params_list
    nll=0
    cues=['GW','NAL','NW','GAL']
    qv_g  = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    qv_ng = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}
    sv    = {'GW': 0.0, 'NAL': 0.0, 'NW': 0.0, 'GAL': 0.0}    
    for  idx, row in df.iterrows():
        cue=row['Trial_Type']
        pressed = int(row['Agent_Choice'])    
        
        wv_g  = qv_g[cue] + b + pi * sv[cue]
        wv_ng = qv_ng[cue]
        # convert the diff into a probability using sigmoid
        pGo = inv_logit(wv_g - wv_ng) 
        pGo = pGo*(1-xi) # scale down by noise rate
        pGo = pGo+(xi/2) # add back random noise floor
        if pressed: # means the pressed is yes, or value =1
            likelihood = pGo
        else:
            likelihood = 1 -pGo
        likelihood = np.clip(likelihood, 1e-10, 1 - 1e-10)
        nll-=np.log(likelihood)
        
        outcome = row['Agent_Reward']
        sv[cue]=sv[cue] +ep *(rho*outcome -sv[cue])
        if (pressed):
            qv_g[cue]= qv_g[cue] +ep *(rho*outcome -qv_g[cue])
        else:
            qv_ng[cue]= qv_ng[cue] +ep *(rho*outcome -qv_ng[cue])
    return nll

In [10]:
x0=[0.1,0.2, 0.3, 1.0, 2.0]
bounds = [
    (0.001, 0.99),   # xi
    (0.001, 0.99),   # ep
    (-5, 5),         # b
    (-5, 5),         # pi
    (0.01, 10),      # rho
]
result = minimize(compute_nll, x0, args=(df,), method='L-BFGS-B',bounds=bounds)


In [11]:
print(f"        recovered params: {result.x.round(2)}")
print(f"param values: {params.values()}")
print(f"model params: {params.keys()}")

        recovered params: [0.1  0.16 0.56 1.21 2.02]
param values: dict_values([0.1, 0.15, 0.5, 1.2, 2.0])
model params: dict_keys(['xi', 'ep', 'b', 'pi', 'rho'])
